# Bank of Canada Valet API - Series Catalog Exploration

## Objective

The goal of this notebook is to explore the Bank of Canada Valet API series catalog.

This first exploration phase aims to:

- Understand the structure of the available financial series
- Analyze metadata quality
- Identify relevant economic and financial indicators
- Select candidate series for the Canadian Financial Lakehouse project

The dataset contains more than 15,000 available time series.

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path



pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

## Load Series Catalog

The series catalog was extracted from the Bank of Canada Valet API.

Each record represents one available economic or financial time series.

Main attributes:

- `series_id`: Unique identifier of the series
- `label`: Short title of the series
- `description`: Detailed description
- `link`: API documentation URL

In [3]:
catalog_path = "../discovery/outputs/series_catalog.csv"

df = pd.read_csv(catalog_path)

df.head()

,series_id,label,description,link
0,A.AGRI,Annual BCPI Agriculture,Annual Bank of Canada commodity price index - ...,https://www.bankofcanada.ca/valet/series/A.AGRI
1,A.BCNE,Annual BCPI Excluding Energy,Annual Bank of Canada commodity price index - ...,https://www.bankofcanada.ca/valet/series/A.BCNE
2,A.BCPI,Annual BCPI Total,Annual Bank of Canada commodity price index - ...,https://www.bankofcanada.ca/valet/series/A.BCPI
3,A.ENER,Annual BCPI Energy,Annual Bank of Canada commodity price index - ...,https://www.bankofcanada.ca/valet/series/A.ENER
4,A.FISH,Annual BCPI Fish,Annual Bank of Canada commodity price index - ...,https://www.bankofcanada.ca/valet/series/A.FISH


## Dataset Overview

Before selecting financial indicators, we need to understand the size and structure of the catalog.

This step helps identify:

- Number of available series
- Available metadata fields
- Potential missing information

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 15906 entries, 0 to 15905
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   series_id    15906 non-null  str  
 1   label        15904 non-null  str  
 2   description  10794 non-null  str  
 3   link         15906 non-null  str  
dtypes: str(4)
memory usage: 497.2 KB


## Initial Observation

The catalog contains:

- 15,906 financial and economic series
- 4 metadata columns
- Complete identifiers and API links
- Some missing descriptions

The `series_id` column will be used as the main reference key during the ingestion phase.

The missing descriptions will be analyzed before applying keyword-based discovery methods.

## Missing Values Analysis

Understanding missing values is an important step in the data discovery phase.

Before using the catalog for automatic series selection, we need to evaluate the completeness of the metadata.

This analysis focuses on:

- Missing values by column
- Data availability for descriptions and labels
- Potential impact on the discovery process

In [7]:
missing_values = df.isnull().sum()

missing_values

series_id         0
label             2
description    5112
link              0
dtype: int64

## Missing Values Percentage

To better understand the impact of missing information, we calculate the percentage of missing values for each column.

This metric provides a normalized view of data completeness regardless of dataset size.

In [8]:
missing_percentage = (
    df.isnull().sum()
    / len(df)
    * 100
)

missing_percentage.round(2)

series_id       0.00
label           0.01
description    32.14
link            0.00
dtype: float64

## Missing Values Observation

The missing values analysis shows that:

- `series_id` and `link` columns are complete with no missing records.
- The `label` column contains only 2 missing values, which has a negligible impact.
- The `description` column contains 5,112 missing values.

The missing descriptions represent approximately one third of the catalog.

Since the `description` field is important for keyword-based financial discovery, we will create additional exploration methods using both `label` and `description` fields.

# Exploring Series Labels and Descriptions

The Bank of Canada catalog contains thousands of economic and financial indicators.

Before selecting specific datasets, we need to understand the terminology used in the series metadata.

This exploration will help identify:

- Main financial domains
- Common keywords
- Potential indicators for the Canadian Financial Lakehouse

In [9]:
df[['series_id', 'label', 'description']].head(20)

,series_id,label,description
0,A.AGRI,Annual BCPI Agriculture,Annual Bank of Canada commodity price index - Agriculture
1,A.BCNE,Annual BCPI Excluding Energy,Annual Bank of Canada commodity price index - Excluding Energy
2,A.BCPI,Annual BCPI Total,Annual Bank of Canada commodity price index - Total
3,A.ENER,Annual BCPI Energy,Annual Bank of Canada commodity price index - Energy
4,A.FISH,Annual BCPI Fish,Annual Bank of Canada commodity price index - Fish
5,A.FOPR,Annual BCPI Forestry,Annual Bank of Canada commodity price index - Forestry
6,A.MTLS,Annual BCPI Metals and Minerals,Annual Bank of Canada commodity price index - Metals and Minerals
7,ABOVE3,Above 3 per cent (% of firms),"Inflation Expectations - Over the next two years, what do you expect the annual rate of inflation to be, based on the consumer price index?"
8,ACTUAL,ACTUAL,Lynx settlement balances - actual
9,AR_2023_C1_S1,2023,


## Initial Series Catalog Observations

The first records show that the Bank of Canada catalog contains different categories of financial and economic information.

Observed categories include:

- Commodity price indexes (BCPI)
- Inflation indicators
- Core inflation measures
- Bond auction information
- Settlement and operational data

Some series are highly relevant for economic analysis, while others are operational or administrative datasets.

A filtering strategy will be required to identify the most valuable indicators for the Canadian Financial Lakehouse.

## Analyze Series Identifier Structure

The `series_id` field appears to follow a naming convention.

The first part of the identifier may indicate the observation frequency.

Examples:

- `A.BCPI` → Annual commodity price index
- `M.xxx` → Monthly series
- `D.xxx` → Daily series

We will extract this information to better understand the catalog structure.

In [11]:
df["frequency"] = (
    df["series_id"]
    .str.split(".")
    .str[0]
)

df["frequency"].value_counts().head(20)

frequency
WGTS         9
A            7
BD           7
M            7
W            7
TB           5
CDN          4
CP           3
BA           2
EUROCAE01    2
IEXE0301     2
IEXE0701     2
IEXE0901     2
IEXE1001     2
IEXE1101     2
IEXE1201     2
IEXE1401     2
IEXE1601     2
IEXE1901     2
IEXE2001     2
Name: count, dtype: int64

## Series Identifier Structure Observation

The initial analysis shows that the `series_id` naming convention is not fully standardized.

Some identifiers contain frequency information:

- `A.xxx` appears to represent annual series
- `M.xxx` appears to represent monthly series
- `D.xxx` appears to represent daily series

However, many identifiers follow different naming patterns.

Therefore, frequency information should not be extracted only from the `series_id` field.

Additional metadata exploration will be required to correctly classify the available series.

In [13]:
common_frequency = (
    df["frequency"]
    .isin(
        [
            "A",
            "M",
            "Q",
            "D",
            "W"
        ]
    )
)

df[common_frequency]["frequency"].value_counts()

frequency
A    7
M    7
W    7
Name: count, dtype: int64

## Frequency Extraction Observation

The analysis shows that only a small number of series follow a simple frequency prefix convention.

Detected standard prefixes:

- `A` : Annual
- `M` : Monthly
- `W` : Weekly

However, this pattern applies to a very limited number of records.

Most Bank of Canada series use other identifier formats, meaning that frequency information cannot reliably be derived from the `series_id` alone.

For the next steps, frequency metadata should be retrieved from the API response or extracted from the original series metadata.

## Prepare Text Fields for Search

The catalog contains missing descriptions.

To perform keyword-based exploration, we create a cleaned text field by combining:

- Series label
- Series description

This allows us to search across all available metadata.

In [14]:
df["search_text"] = (
    df["label"]
    .fillna("")
    + " "
    + df["description"]
    .fillna("")
)

df[["series_id", "search_text"]].head()

,series_id,search_text
0,A.AGRI,Annual BCPI Agriculture Annual Bank of Canada commodity price index - Agriculture
1,A.BCNE,Annual BCPI Excluding Energy Annual Bank of Canada commodity price index - Excluding Energy
2,A.BCPI,Annual BCPI Total Annual Bank of Canada commodity price index - Total
3,A.ENER,Annual BCPI Energy Annual Bank of Canada commodity price index - Energy
4,A.FISH,Annual BCPI Fish Annual Bank of Canada commodity price index - Fish


## Text Preparation Observation

The new `search_text` field combines available information from both labels and descriptions.

This approach improves the discovery process because:

- Some series have detailed descriptions but short labels.
- Some series only provide meaningful information in the label.
- Missing descriptions are handled automatically.

The combined text field will be used for keyword-based financial indicator discovery.

# Keyword-Based Series Search

To identify relevant datasets for the Canadian Financial Lakehouse, we will search the catalog using financial and economic keywords.

This function allows us to quickly explore available indicators related to specific domains.

In [15]:
def search_series(keyword):

    result = df[
        df["search_text"]
        .str.contains(
            keyword,
            case=False,
            na=False
        )
    ]

    return result[
        [
            "series_id",
            "label",
            "description",
            "link"
        ]
    ]

## Commodity Price Indicators

The Bank of Canada publishes several commodity price indexes.

We start with commodity-related indicators because they are important for understanding Canadian economic exposure to natural resources.

In [16]:
commodity_series = search_series("commodity")

commodity_series.head(20)

,series_id,label,description,link
0,A.AGRI,Annual BCPI Agriculture,Annual Bank of Canada commodity price index - Agriculture,https://www.bankofcanada.ca/valet/series/A.AGRI
1,A.BCNE,Annual BCPI Excluding Energy,Annual Bank of Canada commodity price index - Excluding Energy,https://www.bankofcanada.ca/valet/series/A.BCNE
2,A.BCPI,Annual BCPI Total,Annual Bank of Canada commodity price index - Total,https://www.bankofcanada.ca/valet/series/A.BCPI
3,A.ENER,Annual BCPI Energy,Annual Bank of Canada commodity price index - Energy,https://www.bankofcanada.ca/valet/series/A.ENER
4,A.FISH,Annual BCPI Fish,Annual Bank of Canada commodity price index - Fish,https://www.bankofcanada.ca/valet/series/A.FISH
5,A.FOPR,Annual BCPI Forestry,Annual Bank of Canada commodity price index - Forestry,https://www.bankofcanada.ca/valet/series/A.FOPR
6,A.MTLS,Annual BCPI Metals and Minerals,Annual Bank of Canada commodity price index - Metals and Minerals,https://www.bankofcanada.ca/valet/series/A.MTLS
4522,FSR_2019_C12_S01,"Debt at risk, excluding five commodity-related sectors",%,https://www.bankofcanada.ca/valet/series/FSR_2019_C12_S01
4615,FSR_2020_C11_S1,Commodity-related sectors,NaN,https://www.bankofcanada.ca/valet/series/FSR_2020_C11_S1
4725,FSR_2022_C11_S2,Businesses in other commodity industries,NaN,https://www.bankofcanada.ca/valet/series/FSR_2022_C11_S2


## Commodity Indicators Observation

The commodity keyword search identifies several groups of indicators.

The most relevant series for the Canadian Financial Lakehouse are the Bank of Canada Commodity Price Index (BCPI) indicators:

- Total commodity index
- Energy index
- Agriculture index
- Metals and minerals index
- Forestry index

The search also returns financial stability indicators related to commodity sectors.

For the first version of the lakehouse, the focus will be on core economic indicators rather than sector-specific financial reports.

# Financial Domain Exploration

To build a financial data lakehouse, we need to identify the main economic domains available in the Bank of Canada catalog.

We will evaluate several categories:

- Inflation
- Interest rates
- Exchange rates
- Economic growth
- Labour market
- Commodities
- Fixed income markets

In [17]:
keywords = [
    "inflation",
    "consumer price",
    "interest rate",
    "exchange rate",
    "gross domestic",
    "employment",
    "unemployment",
    "commodity",
    "bond"
]


domain_counts = {}

for keyword in keywords:
    count = search_series(keyword).shape[0]
    domain_counts[keyword] = count


pd.Series(domain_counts).sort_values(
    ascending=False
)

inflation         765
bond              678
consumer price    452
exchange rate     407
gross domestic    292
interest rate     285
employment        258
unemployment      179
commodity          35
dtype: int64

## Financial Domain Exploration Results

The keyword-based exploration provides an initial view of the economic domains available in the Bank of Canada catalog.

The largest groups identified are:

- Inflation indicators
- Bond and fixed-income markets
- Consumer price indexes
- Foreign exchange indicators
- Economic activity indicators
- Labour market statistics

These domains represent the main building blocks for a Canadian macro-financial data platform.

The next step is to analyze each domain separately and identify the most relevant time series for the first version of the lakehouse.

# Inflation Indicators Exploration

Inflation is one of the key economic indicators for financial analysis.

The Bank of Canada provides multiple inflation-related series:

- Consumer Price Index (CPI)
- Core inflation measures
- Inflation expectations
- Year-over-year changes

We explore the available inflation indicators to identify the most relevant candidates.

In [18]:
inflation_series = search_series("inflation")

inflation_series.head(30)

,series_id,label,description,link
7,ABOVE3,Above 3 per cent (% of firms),"Inflation Expectations - Over the next two years, what do you expect the annual rate of inflation to be, based on the consumer price index?",https://www.bankofcanada.ca/valet/series/ABOVE3
15,ATOM_V41693242,"CPIX, percentage change year-over-year (unadjusted)","A measure of core inflation that excludes eight of the most volatile Consumer Price Index (CPI) components — fruit, vegetables, gasoline, fuel oil, natural gas, mortgage interest, inter-city trans...",https://www.bankofcanada.ca/valet/series/ATOM_V41693242
278,BELOW1,Below 1 per cent (% of firms),"Inflation Expectations - Over the next two years, what do you expect the annual rate of inflation to be, based on the consumer price index?",https://www.bankofcanada.ca/valet/series/BELOW1
958,BOS_BACKGROUNDER_INDICATOR_2026Q2_C3_S3,CPI inflation,CPI inflation,https://www.bankofcanada.ca/valet/series/BOS_BACKGROUNDER_INDICATOR_2026Q2_C3_S3
970,BOS_BACKGROUNDER_MAGNITUDE_2026Q2_C4_S2,"IPPI inflation (year-over-year percentage change, right scale)","IPPI inflation (year-over-year percentage change, right scale)",https://www.bankofcanada.ca/valet/series/BOS_BACKGROUNDER_MAGNITUDE_2026Q2_C4_S2
972,BOS_BACKGROUNDER_MAGNITUDE_2026Q2_C5_S2,"CPI inflation (year-over-year percentage change, right scale)","CPI inflation (year-over-year percentage change, right scale)",https://www.bankofcanada.ca/valet/series/BOS_BACKGROUNDER_MAGNITUDE_2026Q2_C5_S2
1059,BOS_INFNA_AT,No response,"Inflation expectations of firms in the Atlantic region, four-quarter moving average - Over the next two years, what do you expect the annual rate of inflation to be, based on the consumer price in...",https://www.bankofcanada.ca/valet/series/BOS_INFNA_AT
1060,BOS_INFNA_BC,No response,"Inflation expectations of firms in British Columbia, four-quarter moving average - Over the next two years, what do you expect the annual rate of inflation to be, based on the consumer price index?",https://www.bankofcanada.ca/valet/series/BOS_INFNA_BC
1061,BOS_INFNA_CITU,No response,"Inflation expectations of firms in the CITU sector, four-quarter moving average - Over the next two years, what do you expect the annual rate of inflation to be, based on the consumer price index?",https://www.bankofcanada.ca/valet/series/BOS_INFNA_CITU
1062,BOS_INFNA_CPBS,No response,"Inflation expectations of firms in the CPBS sector, four-quarter moving average - Over the next two years, what do you expect the annual rate of inflation to be, based on the consumer price index?",https://www.bankofcanada.ca/valet/series/BOS_INFNA_CPBS


## Inflation Indicators Observation

The inflation keyword search reveals multiple groups of indicators:

### Core Inflation Measures
Examples:

- CPIX year-over-year percentage change
- Core inflation indicators

These are valuable for macroeconomic analysis.

### Inflation Expectations
The catalog contains detailed expectations data:

- By region
- By firm size
- By economic sector

These indicators are useful but may be too granular for the first lakehouse version.

### CPI Inflation Indicators
Several series provide headline inflation measures and year-over-year changes.

For the initial implementation, priority will be given to official and widely used inflation indicators with long historical coverage.

## CPI Series Exploration

The Consumer Price Index (CPI) is one of the most important indicators for economic analysis.

Instead of searching all inflation-related concepts, we focus specifically on CPI-related series to identify core candidates.

In [19]:
cpi_series = df[
    df["search_text"]
    .str.contains(
        "CPI",
        case=False,
        na=False
    )
]


cpi_series[
    [
        "series_id",
        "label",
        "description"
    ]
].head(30)

,series_id,label,description
0,A.AGRI,Annual BCPI Agriculture,Annual Bank of Canada commodity price index - Agriculture
1,A.BCNE,Annual BCPI Excluding Energy,Annual Bank of Canada commodity price index - Excluding Energy
2,A.BCPI,Annual BCPI Total,Annual Bank of Canada commodity price index - Total
3,A.ENER,Annual BCPI Energy,Annual Bank of Canada commodity price index - Energy
4,A.FISH,Annual BCPI Fish,Annual Bank of Canada commodity price index - Fish
5,A.FOPR,Annual BCPI Forestry,Annual Bank of Canada commodity price index - Forestry
6,A.MTLS,Annual BCPI Metals and Minerals,Annual Bank of Canada commodity price index - Metals and Minerals
15,ATOM_V41693242,"CPIX, percentage change year-over-year (unadjusted)","A measure of core inflation that excludes eight of the most volatile Consumer Price Index (CPI) components — fruit, vegetables, gasoline, fuel oil, natural gas, mortgage interest, inter-city trans..."
958,BOS_BACKGROUNDER_INDICATOR_2026Q2_C3_S3,CPI inflation,CPI inflation
972,BOS_BACKGROUNDER_MAGNITUDE_2026Q2_C5_S2,"CPI inflation (year-over-year percentage change, right scale)","CPI inflation (year-over-year percentage change, right scale)"


## CPI Indicators Selection

The CPI exploration identified several important inflation indicators.

The most relevant candidates for the first version of the Canadian Financial Lakehouse are:

### Headline Inflation

- `INDINF_CPI_M` - Monthly CPI inflation
- `INDINF_CPI_Q` - Quarterly CPI inflation

### Core Inflation Measures

- `CPI_COMMON`
- `CPI_MEDIAN`
- `CPI_TRIM`
- `CPIW`
- `ATOM_V41693242`

These indicators provide complementary views of inflation dynamics:

- Overall price changes
- Core inflation trends
- Measures excluding extreme price movements

The selected indicators will be added to the initial series selection dataset.

# Create Initial Selected Series Metadata

The objective of this step is to create a curated list of series that will be ingested later.

This selection layer separates:

- Available API catalog
- Business-relevant indicators

The selected series will become the foundation of the Bronze ingestion layer.

## 7. Interest Rates and Policy Rate Exploration

Interest rates are a core component of financial and banking analysis.
The Bank of Canada publishes several interest rate indicators, including:

- The policy interest rate (overnight rate target)
- Government bond yields (T-bills, benchmark bonds)
- Prime rate and other lending rates

We explore the catalog to identify the most relevant interest rate series
for the Canadian Financial Lakehouse.

In [20]:
# Search for interest rate related series using multiple targeted keywords
interest_keywords = [
    "overnight",
    "policy rate",
    "prime rate",
    "bank rate",
    "treasury bill",
    "yield"
]

interest_domain_counts = {}
for keyword in interest_keywords:
    count = search_series(keyword).shape[0]
    interest_domain_counts[keyword] = count

pd.Series(interest_domain_counts).sort_values(ascending=False)

policy rate      154
yield            121
treasury bill     56
overnight         36
prime rate         5
bank rate          4
dtype: int64

### 7.1 Overnight Rate and Policy Rate Deep Dive

The overnight rate is the Bank of Canada's key policy interest rate.
It directly influences prime lending rates, mortgage rates, and short-term
borrowing costs across the Canadian financial system.

We isolate the series related to the overnight rate and policy rate target
to identify the primary reference series.

In [21]:
# Deep dive into overnight rate / policy rate series
overnight_series = df[
    df["search_text"]
    .str.contains(
        "overnight rate|policy rate|policy interest rate",
        case=False,
        na=False,
        regex=True
    )
]

overnight_series[
    [
        "series_id",
        "label",
        "description",
        "link"
    ]
].head(30)

,series_id,label,description,link
1263,CBC20210,V39079,Target for the overnight rate,https://www.bankofcanada.ca/valet/series/CBC20210
4755,FSR_2022_C1A_S3,Policy interest rate,NaN,https://www.bankofcanada.ca/valet/series/FSR_2022_C1A_S3
7612,SABA_TROS20260323_C1_S1,Policy interest rates,Policy interest rates,https://www.bankofcanada.ca/valet/series/SABA_TROS20260323_C1_S1
7615,SABA_TROS20260323_C2A_S2,Expectations for policy interest rate,Expectations for policy interest rate,https://www.bankofcanada.ca/valet/series/SABA_TROS20260323_C2A_S2
7618,SABA_TROS20260323_C2B_S2,Expectations for policy interest rate,Expectations for policy interest rate,https://www.bankofcanada.ca/valet/series/SABA_TROS20260323_C2B_S2
7627,SABA_WANG20260330_C1_S1,Change in policy interest rate (left scale),Change in policy interest rate (left scale),https://www.bankofcanada.ca/valet/series/SABA_WANG20260330_C1_S1
8666,SAN_FONT20250113_C4_S1,Bank of Canada (BoC) overnight rate (left scale),NaN,https://www.bankofcanada.ca/valet/series/SAN_FONT20250113_C4_S1
8685,SAN_FONT20250113_C8_S3,Differential between the policy rates of the Bank of Canada and Federal Reserve Board (right scale),NaN,https://www.bankofcanada.ca/valet/series/SAN_FONT20250113_C8_S3
8998,SAN_LEBM180118_CHART2F_A_OR,OR,Impulse response for overnight rate,https://www.bankofcanada.ca/valet/series/SAN_LEBM180118_CHART2F_A_OR
8999,SAN_LEBM180118_CHART2F_B_OR_LOWER,OR_Lower,Impulse response lower confidence bound for overnight rate,https://www.bankofcanada.ca/valet/series/SAN_LEBM180118_CHART2F_B_OR_LOWER


### 7.2 Overnight Rate Reference Series Identified

The search results show that most matches are research notes, staff analytical
notes, or projection series (prefixes such as `SAN_`, `SABA_`, `SWP-`), which
are not suitable for time series ingestion.

The relevant reference series is:

- **CBC20210** (V39079) — *Target for the overnight rate*

This is the Bank of Canada's official policy interest rate and a top-priority
candidate for the Canadian Financial Lakehouse.

A related static/legacy series (`STATIC_ATABLE_V39079`) also appears, likely
representing an older table format for the same underlying data (V39079).
We will keep only the primary series and treat the static version as a
duplicate to exclude.

In [22]:
# Confirm the overnight rate reference series and check for duplicate/legacy versions
overnight_rate_reference = df[
    df["series_id"].isin(["CBC20210", "STATIC_ATABLE_V39079"])
]

overnight_rate_reference[
    [
        "series_id",
        "label",
        "description",
        "link"
    ]
]

,series_id,label,description,link
1263,CBC20210,V39079,Target for the overnight rate,https://www.bankofcanada.ca/valet/series/CBC20210
9841,STATIC_ATABLE_V39079,Overnight rate target (end of month),Overnight rate target (end of month),https://www.bankofcanada.ca/valet/series/STATIC_ATABLE_V39079


## 8. Government Bond Yields and Treasury Bills Exploration

Government bond yields and treasury bill rates are essential indicators for
fixed-income analysis and are widely used as benchmarks across the Canadian
banking and financial sector.

We explore the catalog to identify:

- Treasury bill (T-bill) reference rates
- Benchmark government bond yields (short, medium, and long-term)

In [23]:
# Search for treasury bill and government bond yield series
bond_keywords = [
    "treasury bill",
    "government of canada bond",
    "benchmark bond",
    "bond yield"
]

bond_domain_counts = {}
for keyword in bond_keywords:
    count = search_series(keyword).shape[0]
    bond_domain_counts[keyword] = count

pd.Series(bond_domain_counts).sort_values(ascending=False)

government of canada bond    59
treasury bill                56
bond yield                   26
benchmark bond               15
dtype: int64

### 8.1 Benchmark Bond Yields and Treasury Bill Deep Dive

Government of Canada bond yields and treasury bill rates are typically
published as standard benchmark maturities (e.g., 3-month T-bill, 2-year,
5-year, 10-year, and long-term bond yields).

We isolate the series matching these standard maturities to identify the
primary reference series for the fixed-income domain.

In [24]:
# Deep dive into benchmark bond and treasury bill series
bond_series = df[
    df["search_text"]
    .str.contains(
        "benchmark bond|treasury bill|government of canada bond",
        case=False,
        na=False,
        regex=True
    )
]

bond_series[
    [
        "series_id",
        "label",
        "description",
        "link"
    ]
].head(40)

,series_id,label,description,link
271,BD.CDN.10YR.DQ.YLD,Benchmark bond yield: 10 year,The yield on the Government of Canada 10-year benchmark bond. This series represents the prevailing market interest rate for the 10-year Government of Canada bond and is used as a reference for fi...,https://www.bankofcanada.ca/valet/series/BD.CDN.10YR.DQ.YLD
272,BD.CDN.2YR.DQ.YLD,Benchmark bond yield: 2 year,The yield on the Government of Canada 2-year benchmark bond. This series reflects the prevailing interest rate for the 2-year term and is used to monitor short-term government borrowing costs.,https://www.bankofcanada.ca/valet/series/BD.CDN.2YR.DQ.YLD
273,BD.CDN.3YR.DQ.YLD,Benchmark bond yield: 3 year,"The yield on the Government of Canada 3-year benchmark bond. This series measures the market interest rate for the Government of Canada’s 3-year bond, which serves as a reference rate for Canadian...",https://www.bankofcanada.ca/valet/series/BD.CDN.3YR.DQ.YLD
274,BD.CDN.5YR.DQ.YLD,Benchmark bond yield: 5 year,The yield on the Government of Canada 5-year benchmark bond. This series reports the prevailing yield for the 5-year maturity point on the Canadian government bond yield curve.,https://www.bankofcanada.ca/valet/series/BD.CDN.5YR.DQ.YLD
275,BD.CDN.7YR.DQ.YLD,Benchmark bond yield: 7 year,The yield on the Government of Canada 7-year benchmark bond. This series represents the prevailing market yield for the 7-year maturity point on the Government of Canada yield curve.,https://www.bankofcanada.ca/valet/series/BD.CDN.7YR.DQ.YLD
276,BD.CDN.LONG.DQ.YLD,Benchmark bond yield: long-term,The yield on the Government of Canada long-term benchmark bond. This series represents the prevailing interest rate for the designated long-term benchmark bond and is used in the Bank of Canada’s ...,https://www.bankofcanada.ca/valet/series/BD.CDN.LONG.DQ.YLD
279,BMDBGOCT.DALL,Banks,Government of Canada bonds - Domestic,https://www.bankofcanada.ca/valet/series/BMDBGOCT.DALL
280,BMDIBGOCT.DALL,Anonymous systems,Government of Canada bonds - Domestic,https://www.bankofcanada.ca/valet/series/BMDIBGOCT.DALL
281,BMDIDGOCT.DALL,Investment dealers,Government of Canada bonds - Domestic,https://www.bankofcanada.ca/valet/series/BMDIDGOCT.DALL
282,BMDOGOCT.DALL,Other,Government of Canada bonds - Domestic,https://www.bankofcanada.ca/valet/series/BMDOGOCT.DALL


### 8.2 Benchmark Bond Yield Reference Series Identified

The series matching the `BD.CDN.*.DQ.YLD` naming pattern represent the
official Government of Canada benchmark bond yields across standard
maturities. These follow a consistent, well-structured naming convention
and are ideal candidates for ingestion.

Identified benchmark maturities:

- **BD.CDN.2YR.DQ.YLD** — 2-year
- **BD.CDN.3YR.DQ.YLD** — 3-year
- **BD.CDN.5YR.DQ.YLD** — 5-year
- **BD.CDN.7YR.DQ.YLD** — 7-year
- **BD.CDN.10YR.DQ.YLD** — 10-year
- **BD.CDN.LONG.DQ.YLD** — Long-term

Other series identified in this search fall into different categories:

- `BM*` series (e.g., `BMTGOCT.DALL`) — bond market trading volumes by
  maturity bucket and holder type. Useful for market activity analysis but
  secondary for the first lakehouse version.
- `FSR_*` and `SAN_*` series — one-off charts from Financial System Review
  reports or staff analytical notes, not suitable for time series ingestion.
- `FVI_*` series — financial vulnerability indicators (term premium, bid-ask
  spread, yield volatility). Interesting for a more advanced version but out
  of scope for the initial MVP.

We now confirm the T-bill reference rate separately, since treasury bills
did not appear in the standardized `BD.CDN.*` naming convention above.

In [25]:
# Verify benchmark bond yield series and search separately for T-bill reference rate
benchmark_bonds = df[
    df["series_id"].str.contains(r"^BD\.CDN\..*\.DQ\.YLD$", regex=True, na=False)
]

tbill_reference = df[
    df["search_text"].str.contains("treasury bill", case=False, na=False)
    & df["series_id"].str.contains(r"^TB\.|^BD\.", regex=True, na=False)
]

print("Benchmark bonds:")
display(benchmark_bonds[["series_id", "label", "description"]])

print("\nT-bill candidates:")
display(tbill_reference[["series_id", "label", "description"]])

Benchmark bonds:


,series_id,label,description
271,BD.CDN.10YR.DQ.YLD,Benchmark bond yield: 10 year,The yield on the Government of Canada 10-year benchmark bond. This series represents the prevailing market interest rate for the 10-year Government of Canada bond and is used as a reference for fi...
272,BD.CDN.2YR.DQ.YLD,Benchmark bond yield: 2 year,The yield on the Government of Canada 2-year benchmark bond. This series reflects the prevailing interest rate for the 2-year term and is used to monitor short-term government borrowing costs.
273,BD.CDN.3YR.DQ.YLD,Benchmark bond yield: 3 year,"The yield on the Government of Canada 3-year benchmark bond. This series measures the market interest rate for the Government of Canada’s 3-year bond, which serves as a reference rate for Canadian..."
274,BD.CDN.5YR.DQ.YLD,Benchmark bond yield: 5 year,The yield on the Government of Canada 5-year benchmark bond. This series reports the prevailing yield for the 5-year maturity point on the Canadian government bond yield curve.
275,BD.CDN.7YR.DQ.YLD,Benchmark bond yield: 7 year,The yield on the Government of Canada 7-year benchmark bond. This series represents the prevailing market yield for the 7-year maturity point on the Government of Canada yield curve.
276,BD.CDN.LONG.DQ.YLD,Benchmark bond yield: long-term,The yield on the Government of Canada long-term benchmark bond. This series represents the prevailing interest rate for the designated long-term benchmark bond and is used in the Bank of Canada’s ...
277,BD.CDN.RRB.DQ.YLD,Real Return Bonds yield: long-term,The yield on Government of Canada Real Return Bonds. These bonds are part of the government securities held by the Bank of Canada and are reported in millions of Canadian dollars.



T-bill candidates:


,series_id,label,description
12597,TB.CDN.180D.MID,V39066,Treasury bills - 6 month
12598,TB.CDN.1Y.MID,V39067,Treasury bills - 1 year
12599,TB.CDN.30D.MID,V39063,Treasury bills - 1 month
12600,TB.CDN.60D.MID,V39064,Treasury bills - 2 month
12601,TB.CDN.90D.MID,V39065,Treasury bills - 3 month


### 8.3 Fixed Income Reference Series — Summary

Two clean, well-structured series families have been identified for the
fixed-income domain:

**Government of Canada benchmark bond yields** (`BD.CDN.*.DQ.YLD`):
- 2, 3, 5, 7, 10-year, long-term, and Real Return Bonds (RRB)

**Government of Canada treasury bills** (`TB.CDN.*.MID`):
- 1, 2, 3, 6-month, and 1-year maturities

Both families follow a consistent naming convention, which simplifies
ingestion logic (a single loop over maturities rather than series-specific
handling). These 12 series form the core fixed-income reference set for
the Canadian Financial Lakehouse.

We now move to the foreign exchange domain to identify the primary
USD/CAD and other currency reference rates.

In [26]:
# Explore foreign exchange rate series
fx_series = df[
    df["search_text"].str.contains(
        "exchange rate|FX", case=False, na=False, regex=True
    )
]

print(f"Total FX-related series: {fx_series.shape[0]}")

fx_series[
    ["series_id", "label", "description"]
].head(30)

Total FX-related series: 407


,series_id,label,description
1290,CEER_BROADM,Monthly real Canadian effective exchange rates,"The monthly value of the real Canadian effective exchange rate (CEER) index, calculated using a weighted average of bilateral exchange rates against a basket of major trading partners’ currencies...."
1291,CEER_BROADN,Daily nominal Canadian effective exchange rates,"The daily value of the nominal Canadian effective exchange rate (CEER) index, calculated by using a weighted average of bilateral exchange rates against a basket of major trading partners’ currenc..."
1292,CEER_BROADNM,Monthly nominal Canadian effective exchange rates,"The monthly value of the nominal Canadian effective exchange rate (CEER) index, calculated using a weighted average of bilateral exchange rates against a basket of major trading partners’ currenci..."
1293,CEER_BROADN_XUS,Daily nominal CEER excluding the US dollar,"The daily value of the nominal Canadian effective exchange rate (CEER) index, excluding the US dollar from the basket of currencies. The index is set to 100 for the base year 1999."
1294,CEER_BROADN_XUSM,Monthly nominal CEER excluding the US dollar,"The monthly value of the nominal Canadian effective exchange rate (CEER) index, excluding the US dollar from the basket of currencies. The index is set to 100 for the base year 1999."
1295,CEER_BROAD_XUSM,Monthly real CEER excluding the US dollar,"The monthly value of the real Canadian effective exchange rate (CEER) index, excluding the US dollar from the basket of currencies. The index is set to 100 for the base year 1999."
1296,CEER_MCM,Monthly real CEER versus major currencies,"The monthly value of the real Canadian effective exchange rate (CEER) index, calculated against a basket of major currencies. The index is set to 100 for the base year 1999."
1297,CEER_MCN,Daily nominal CEER versus major currencies,"The daily value of the nominal Canadian effective exchange rate (CEER) index, calculated against a basket of major currencies. The index is set to 100 for the base year 1999."
1298,CEER_MCNM,Monthly nominal CEER versus major currencies,"The monthly value of the nominal Canadian effective exchange rate (CEER) index, calculated against a basket of major currencies. The index is set to 100 for the base year 1999."
1299,CEER_OITPM,Monthly real CEER versus other important trading partners,"The monthly value of the real Canadian effective exchange rate (CEER) index, calculated against the currencies of other important trading partners. The index is set to 100 for the base year 1999."


### 9.1 Daily Exchange Rate Series — USD/CAD Reference

The `FXA*CAD` series represent annual average exchange rates. For a
financial lakehouse, daily exchange rates provide more analytical value
(volatility analysis, time series modeling, correlation with interest rates).

We search specifically for the daily USD/CAD series and check whether the
same naming convention (`FXA`, `FXD`, `FXM`) exists across annual, daily,
and monthly frequencies.

In [27]:
# Search for USD/CAD series across all naming conventions and frequencies
usd_cad_series = df[
    df["series_id"].str.contains("USDCAD", case=False, na=False)
]

usd_cad_series[
    ["series_id", "label", "description"]
]

,series_id,label,description
5461,FXAUSDCAD,USD/CAD,"Annual average exchange rate: annual value of the US dollar expressed in Canadian dollars, for 1 unit of US dollar"
5526,FXMUSDCAD,USD/CAD,"Monthly average exchange rate: monthly value of the US dollar expressed in Canadian dollars, for 1 unit of US dollar"
5542,FXUSDCAD,USD/CAD,"Daily average exchange rate: daily value of the US dollar expressed in Canadian dollars, for 1 unit of US dollar"


### 9.2 Foreign Exchange Naming Convention Confirmed

The FX series follow a consistent prefix convention:

- **FXA{CCY}CAD** — Annual average
- **FXM{CCY}CAD** — Monthly average
- **FX{CCY}CAD** — Daily average (no frequency prefix)

The primary reference series identified is:

- **FXUSDCAD** — Daily USD/CAD exchange rate

This is the single most important FX series for the Canadian Financial
Lakehouse, given the central role of the US dollar in Canadian trade and
financial markets. We now check which other major currencies have a
daily-frequency counterpart, to build the initial FX selection set.

In [28]:
# Identify all daily-frequency FX series (no FXA/FXM prefix) against CAD
daily_fx_series = df[
    df["series_id"].str.match(r"^FX(?!A|M)[A-Z]{3}CAD$", na=False)
]

daily_fx_series[
    ["series_id", "label", "description"]
]

,series_id,label,description
5464,FXBRLCAD,BRL/CAD,"Daily average exchange rate: daily value of the Brazilian real expressed in Canadian dollars, for 1 unit of Brazilian real"
5493,FXCHFCAD,CHF/CAD,"Daily average exchange rate: daily value of the Swiss franc expressed in Canadian dollars, for 1 unit of Swiss franc"
5494,FXCNYCAD,CNY/CAD,"Daily average exchange rate: daily value of the Chinese renminbi expressed in Canadian dollars, for 1 unit of Chinese renminbi"
5495,FXEURCAD,EUR/CAD,"Daily average exchange rate: daily value of the euro expressed in Canadian dollars, for 1 unit of euro"
5496,FXGBPCAD,GBP/CAD,"Daily average exchange rate: daily value of the UK pound sterling expressed in Canadian dollars, for 1 unit of pound sterling"
5497,FXHKDCAD,HKD/CAD,"Daily average exchange rate: daily value of the Hong Kong dollar expressed in Canadian dollars, for 1 unit of Hong Kong dollar"
5498,FXIDRCAD,IDR/CAD,"Daily average exchange rate: daily value of the Indonesian rupiah expressed in Canadian dollars, for 1 unit of Indonesian rupiah"
5499,FXINRCAD,INR/CAD,"Daily average exchange rate: daily value of the Indian rupee expressed in Canadian dollars, for 1 unit of Indian rupee"
5500,FXJPYCAD,JPY/CAD,"Daily average exchange rate: daily value of the Japanese yen expressed in Canadian dollars, for 1 unit of Japanese yen"
5501,FXKRWCAD,KRW/CAD,"Daily average exchange rate: daily value of the South Korean won expressed in Canadian dollars, for 1 unit of South Korean won"


### 9.3 Foreign Exchange Selection Criteria

21 daily FX series against CAD are available. For the initial version of
the lakehouse, we distinguish between:

**Major currencies (high priority — G7 and primary trading partners):**
- USD, EUR, GBP, JPY, CHF, CNY

**Secondary currencies (lower priority — smaller trading volumes):**
- AUD, NZD, HKD, SGD, and other regional currencies

Some series descriptions are explicitly labeled "historical series"
(e.g., RUB, SAR, VND), suggesting these may be discontinued or maintained
only for historical reference rather than active daily updates. These will
be flagged for verification during the ingestion phase (checking last
observation date via the API) rather than excluded outright at this stage.

We now move to the GDP and employment domain to complete the macroeconomic
context before finalizing the banking-specific keyword search.

In [29]:
# Explore GDP and employment/labour market series
macro_keywords = [
    "gross domestic product",
    "unemployment rate",
    "employment rate",
    "labour force"
]

macro_domain_counts = {}
for keyword in macro_keywords:
    count = search_series(keyword).shape[0]
    macro_domain_counts[keyword] = count

pd.Series(macro_domain_counts).sort_values(ascending=False)

gross domestic product    286
employment rate           175
unemployment rate         159
labour force               18
dtype: int64

### 10. GDP and Labour Market — Reference Series

GDP and labour market indicators (unemployment rate, employment rate) are
useful as macroeconomic context for financial analysis, but are secondary
to the core financial indicators (interest rates, bond yields, FX) for this
lakehouse. Rather than a full deep-dive, we identify a small set of
official headline reference series for each theme.

We search specifically for the standard headline series:
- Real GDP (monthly, official Statistics Canada measure via Bank of Canada)
- Unemployment rate (headline, seasonally adjusted)

In [30]:
# Identify headline GDP and unemployment rate reference series
gdp_reference = df[
    df["search_text"].str.contains("real gross domestic product|real GDP", case=False, na=False, regex=True)
]

unemployment_reference = df[
    df["search_text"].str.contains("unemployment rate", case=False, na=False)
    & ~df["search_text"].str.contains("expect|survey|BOS_", case=False, na=False, regex=True)
]

print("GDP candidates:")
display(gdp_reference[["series_id", "label", "description"]].head(15))

print("\nUnemployment rate candidates:")
display(unemployment_reference[["series_id", "label", "description"]].head(15))

GDP candidates:


,series_id,label,description
955,BOS_BACKGROUNDER_INDICATOR_2026Q2_C2_S3,Real GDP growth,Real GDP growth
4670,FSR_2021_C16_S1,Projected level of real GDP in the April 2021 Monetary Policy Report,NaN
4671,FSR_2021_C16_S2,Level of real GDP below which banks would likely face balance sheet pressures to reduce credit,
6075,MPR_2023M07_SQP_CAN_GDP_QOQ,Real GDP (quarter-over-quarter percentage change at annual rates),Real GDP (quarter-over-quarter percentage change at annual rates)
6076,MPR_2023M07_SQP_CAN_GDP_YOY,Real GDP (year-over-year percentage change),Real GDP (year-over-year percentage change)
6091,MPR_2023M10_SQP_CAN_GDP_QOQ,Real GDP (quarter-over-quarter percentage change at annual rates),Real GDP (quarter-over-quarter percentage change at annual rates)
6092,MPR_2023M10_SQP_CAN_GDP_YOY,Real GDP (year-over-year percentage change),Real GDP (year-over-year percentage change)
6107,MPR_2024M01_SQP_CAN_GDP_QOQ,Real GDP (quarter-over-quarter percentage change at annual rates),Real GDP (quarter-over-quarter percentage change at annual rates)
6108,MPR_2024M01_SQP_CAN_GDP_YOY,Real GDP (year-over-year percentage change),Real GDP (year-over-year percentage change)
6138,MPR_2024M04_SQP_CAN_GDP_QOQ,Real GDP (quarter-over-quarter percentage change at annual rates),Real GDP (quarter-over-quarter percentage change at annual rates)



Unemployment rate candidates:


,series_id,label,description
4798,FSR_2023_C11_S1,Unemployment rate (left scale),NaN
6273,MPR_2024M10_2006_S1,Unemployment rate (left scale),NaN
6780,MPR_2025M07_LMI_HEATMAP_S1,Unemployment rate,NaN
6961,MPR_2025M10_LMI_HEATMAP_QUARTERLY_S1,Unemployment rate,NaN
7245,MPR_2026M04_LABOUR_HEATMAP_S1,Unemployment rate,Unemployment rate
7269,MPR_2026M04_US_EMPLOYMENT_S1,Unemployment rate (left scale),Unemployment rate (left scale)
7399,MPR_2026M07_LABOUR_HEATMAP_S1,Unemployment rate,Unemployment rate
8092,SAN_BOUN20240501_C2_S2,Unemployment rate (right scale),NaN
12093,SWP-URATE1982Q2,URATE - Staff Projection 1982Q2,Unemployment rate - Staff Projection 1982Q2
12094,SWP-URATE1982Q3,URATE - Staff Projection 1982Q3,Unemployment rate - Staff Projection 1982Q3


### 10.1 GDP and Unemployment — Data Availability Observation

No stable, continuously updated GDP or unemployment rate series were found
in the Bank of Canada Valet catalog. All matching series are tied to
specific report editions (Monetary Policy Report, Financial System Review,
Business Outlook Survey, staff projections), identifiable by prefixes such
as `MPR_`, `FSR_`, `SWP-`, `BOS_`, and `SAN_`. These are one-off chart data
points published alongside a specific report, not continuous time series
suitable for ingestion.

This is expected: GDP and labour market statistics are produced by
Statistics Canada, not the Bank of Canada. The Valet API occasionally
republishes short excerpts within its own analytical reports, but does not
host the canonical GDP/employment series.

**Decision for the first version of the lakehouse:**
GDP and unemployment rate are excluded from the initial series selection.
They remain a candidate for a future enhancement via the Statistics Canada
Web Data Service (WDS), integrated as a secondary data source outside the
scope of this project's first version.

We now complete the domain exploration with banking-specific keywords
before finalizing the series selection table.

In [31]:
# Search for banking-sector-specific keywords
banking_keywords = [
    "chartered bank",
    "credit",
    "mortgage",
    "deposit",
    "monetary aggregate",
    "reserve"
]

banking_domain_counts = {}
for keyword in banking_keywords:
    count = search_series(keyword).shape[0]
    banking_domain_counts[keyword] = count

pd.Series(banking_domain_counts).sort_values(ascending=False)

mortgage              854
deposit               356
credit                353
chartered bank         24
monetary aggregate      9
reserve                 9
dtype: int64

### 11. Banking Sector Keywords — Headline Series Only

The banking-related keyword search returns a large volume of series,
particularly for "mortgage" (854 matches). A full deep-dive on each keyword
would be disproportionate for a first lakehouse version, since many of
these series are highly granular (broken down by lender type, term,
province, or loan purpose).

For this MVP, we isolate only the headline, aggregate-level series for
each theme:

- Chartered banks' assets/liabilities (aggregate)
- Mortgage rate (posted rate, headline)
- Monetary aggregates (M1, M2, M3)

More granular banking series are noted as candidates for a future
enhancement of the lakehouse rather than the initial version.

In [32]:
# Isolate headline banking series: monetary aggregates and posted mortgage rate
monetary_aggregates = df[
    df["search_text"].str.contains("monetary aggregate", case=False, na=False)
]

mortgage_rate_headline = df[
    df["series_id"].str.contains(r"^V80691|MORTG|CONVMORTG", case=False, na=False, regex=True)
    & df["search_text"].str.contains("posted|conventional", case=False, na=False)
]

print("Monetary aggregates:")
display(monetary_aggregates[["series_id", "label", "description"]])

print("\nPosted mortgage rate candidates:")
display(mortgage_rate_headline[["series_id", "label", "description"]].head(15))

Monetary aggregates:


,series_id,label,description
5978,M1PP_12MTH,M1++ (Year-over-year growth rates),Monetary aggregates
5979,M1PP_ANN_3MTH,M1++ (Annualized 3-month growth rates),Monetary aggregates
5980,M1P_12MTH,M1+ (Year-over-year growth rates),Monetary aggregates
5981,M1P_ANN_3MTH,M1+ (Annualized 3-month growth rates),Monetary aggregates
5982,M2PP_12MTH,M2++ (Year-over-year growth rates),Monetary aggregates
5983,M2PP_ANN_3MTH,M2++ (Annualized 3-month growth rates),Monetary aggregates
9839,STATIC_ATABLE_V37151,M1+ (gross),"This monetary aggregate includes: Currency outside banks plus personal and non-personal chequable deposits held at chartered banks plus all chequable deposits at trust and mortgage loan companies,..."
9840,STATIC_ATABLE_V37152,M1++ (gross),"This monetary aggregate includes: M1+ (gross) plus non-chequable notice deposits held at chartered banks plus all non-chequable deposits at trust and mortgage loan companies, credit unions and cai..."
9842,STATIC_ATABLE_V41552801,M2++ (gross),This monetary aggregate includes: M2+ (gross) plus Canada Savings Bonds and other retail instruments plus cumulative net contributions to mutual funds other than Canadian dollar money market mutua...



Posted mortgage rate candidates:


,series_id,label,description
14613,V80691333,Conventional mortgage: 1-year,The interest rate for a 1-year conventional mortgage offered by chartered banks in Canada.
14614,V80691334,Conventional mortgage: 3-year,The interest rate for a 3-year conventional mortgage offered by chartered banks in Canada.
14615,V80691335,Conventional mortgage: 5-year,The interest rate for a 5-year conventional mortgage offered by chartered banks in Canada.
14616,V80691336,5-year personal fixed term,The posted interest rate for personal deposits with a fixed term of five years at chartered banks in Canada.


## 12. Final Series Selection — Bronze Layer Candidates

Based on the domain exploration above, we now consolidate the selected
series into a single reference table. This table separates:

- The full API catalog (15,906 series)
- The business-relevant indicators selected for the Canadian Financial
  Lakehouse

Each selected series is tagged with its financial domain and a short
justification. This selection table becomes the configuration reference
for the Bronze ingestion layer (Phase 2).

**Selection criteria applied:**
- Official, continuously updated series (excludes one-off report charts
  such as `FSR_`, `MPR_`, `SAN_`, `SWP-`, `BOS_`)
- Consistent naming conventions preferred (enables generic ingestion logic)
- Priority given to headline/aggregate series over granular breakdowns
- GDP and unemployment excluded (not natively available in this API)

In [33]:
# Build the final selected series reference table
selected_series_ids = {
    # Policy interest rate
    "CBC20210": ("Interest Rates", "Target for the overnight rate (Bank of Canada policy rate)"),

    # Government of Canada benchmark bond yields
    "BD.CDN.2YR.DQ.YLD": ("Fixed Income", "Benchmark bond yield: 2 year"),
    "BD.CDN.3YR.DQ.YLD": ("Fixed Income", "Benchmark bond yield: 3 year"),
    "BD.CDN.5YR.DQ.YLD": ("Fixed Income", "Benchmark bond yield: 5 year"),
    "BD.CDN.7YR.DQ.YLD": ("Fixed Income", "Benchmark bond yield: 7 year"),
    "BD.CDN.10YR.DQ.YLD": ("Fixed Income", "Benchmark bond yield: 10 year"),
    "BD.CDN.LONG.DQ.YLD": ("Fixed Income", "Benchmark bond yield: long-term"),

    # Treasury bills
    "TB.CDN.30D.MID": ("Fixed Income", "Treasury bill: 1 month"),
    "TB.CDN.60D.MID": ("Fixed Income", "Treasury bill: 2 month"),
    "TB.CDN.90D.MID": ("Fixed Income", "Treasury bill: 3 month"),
    "TB.CDN.180D.MID": ("Fixed Income", "Treasury bill: 6 month"),
    "TB.CDN.1Y.MID": ("Fixed Income", "Treasury bill: 1 year"),

    # Foreign exchange (major currencies, daily frequency)
    "FXUSDCAD": ("Foreign Exchange", "USD/CAD daily exchange rate"),
    "FXEURCAD": ("Foreign Exchange", "EUR/CAD daily exchange rate"),
    "FXGBPCAD": ("Foreign Exchange", "GBP/CAD daily exchange rate"),
    "FXJPYCAD": ("Foreign Exchange", "JPY/CAD daily exchange rate"),
    "FXCHFCAD": ("Foreign Exchange", "CHF/CAD daily exchange rate"),
    "FXCNYCAD": ("Foreign Exchange", "CNY/CAD daily exchange rate"),

    # Inflation
    "INDINF_CPI_M": ("Inflation", "Monthly CPI inflation"),
    "CPI_COMMON": ("Inflation", "CPI-common (core inflation measure)"),
    "CPI_MEDIAN": ("Inflation", "CPI-median (core inflation measure)"),
    "CPI_TRIM": ("Inflation", "CPI-trim (core inflation measure)"),

    # Monetary aggregates
    "M1PP_12MTH": ("Monetary Aggregates", "M1++ year-over-year growth rate"),
    "M2PP_12MTH": ("Monetary Aggregates", "M2++ year-over-year growth rate"),

    # Mortgage rates
    "V80691333": ("Banking", "Conventional mortgage rate: 1 year"),
    "V80691334": ("Banking", "Conventional mortgage rate: 3 year"),
    "V80691335": ("Banking", "Conventional mortgage rate: 5 year"),
}

selected_df = pd.DataFrame(
    [
        {"series_id": sid, "domain": domain, "justification": note}
        for sid, (domain, note) in selected_series_ids.items()
    ]
)

# Merge with catalog metadata to confirm labels match
selected_df = selected_df.merge(
    df[["series_id", "label", "link"]],
    on="series_id",
    how="left"
)

print(f"Total selected series: {selected_df.shape[0]}")
selected_df[["series_id", "domain", "label", "justification", "link"]]

Total selected series: 27


,series_id,domain,label,justification,link
0,CBC20210,Interest Rates,V39079,Target for the overnight rate (Bank of Canada policy rate),https://www.bankofcanada.ca/valet/series/CBC20210
1,BD.CDN.2YR.DQ.YLD,Fixed Income,Benchmark bond yield: 2 year,Benchmark bond yield: 2 year,https://www.bankofcanada.ca/valet/series/BD.CDN.2YR.DQ.YLD
2,BD.CDN.3YR.DQ.YLD,Fixed Income,Benchmark bond yield: 3 year,Benchmark bond yield: 3 year,https://www.bankofcanada.ca/valet/series/BD.CDN.3YR.DQ.YLD
3,BD.CDN.5YR.DQ.YLD,Fixed Income,Benchmark bond yield: 5 year,Benchmark bond yield: 5 year,https://www.bankofcanada.ca/valet/series/BD.CDN.5YR.DQ.YLD
4,BD.CDN.7YR.DQ.YLD,Fixed Income,Benchmark bond yield: 7 year,Benchmark bond yield: 7 year,https://www.bankofcanada.ca/valet/series/BD.CDN.7YR.DQ.YLD
5,BD.CDN.10YR.DQ.YLD,Fixed Income,Benchmark bond yield: 10 year,Benchmark bond yield: 10 year,https://www.bankofcanada.ca/valet/series/BD.CDN.10YR.DQ.YLD
6,BD.CDN.LONG.DQ.YLD,Fixed Income,Benchmark bond yield: long-term,Benchmark bond yield: long-term,https://www.bankofcanada.ca/valet/series/BD.CDN.LONG.DQ.YLD
7,TB.CDN.30D.MID,Fixed Income,V39063,Treasury bill: 1 month,https://www.bankofcanada.ca/valet/series/TB.CDN.30D.MID
8,TB.CDN.60D.MID,Fixed Income,V39064,Treasury bill: 2 month,https://www.bankofcanada.ca/valet/series/TB.CDN.60D.MID
9,TB.CDN.90D.MID,Fixed Income,V39065,Treasury bill: 3 month,https://www.bankofcanada.ca/valet/series/TB.CDN.90D.MID


### 12.1 Final Selection Summary

The final selection contains **27 series** across 6 financial domains:

| Domain | Series count |
|---|---|
| Fixed Income | 11 |
| Foreign Exchange | 6 |
| Inflation | 4 |
| Interest Rates | 1 |
| Monetary Aggregates | 2 |
| Banking | 3 |

All selected series matched successfully against the catalog metadata with
no missing labels, confirming that the `series_id` values are correctly
referenced.

This selection represents the initial candidate set for Bronze layer
ingestion. It intentionally excludes GDP, unemployment, and granular
sector-specific series, which are candidates for future iterations of the
lakehouse.

The selection table is exported below as the configuration reference for
the ingestion phase.

In [34]:
# Export the final selection table for use in the ingestion phase
output_path = Path("../discovery/outputs/selected_series.csv")
selected_df.to_csv(output_path, index=False)

print(f"Selected series exported to: {output_path}")
print(f"Total series: {selected_df.shape[0]}")
selected_df["domain"].value_counts()

Selected series exported to: ../discovery/outputs/selected_series.csv
Total series: 27


domain
Fixed Income           11
Foreign Exchange        6
Inflation               4
Banking                 3
Monetary Aggregates     2
Interest Rates          1
Name: count, dtype: int64